# 07c — Caracterización combinada (Nivel 1 × Nivel 2)


In [1]:
import pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '05_archetypes'

assignments = pd.read_parquet(DATA_QC / 'two_stage_assignments.parquet')
print(f'Assignments: {assignments.shape}')

# Crosstab: row-normalized (% dentro de cada arquetipo identitario)
assignments['n2_cluster'] = assignments['nivel2_cluster'].fillna(-99).astype(int)
# -99 = sin Tier 2, -1 = outlier en Tier 2, 0-4 = sub-clusters
ct = pd.crosstab(assignments['nivel1_cluster'], assignments['n2_cluster'], normalize='index') * 100
ct_n = pd.crosstab(assignments['nivel1_cluster'], assignments['n2_cluster'])

print('Crosstab (%):')
print(ct.round(1).to_string())

ct.to_csv(INFORMES / 'nivel1_x_nivel2_crosstab_pct.csv')
ct_n.to_csv(INFORMES / 'nivel1_x_nivel2_crosstab_counts.csv')

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
sns.heatmap(ct, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label':'% dentro de N1'})
axes[0].set_title('Nivel 1 × Nivel 2 (% por fila)')
axes[0].set_xlabel('Nivel 2 cluster (-99=sin Tier2, -1=outlier)')
axes[0].set_ylabel('Nivel 1 cluster')
sns.heatmap(ct_n, annot=True, fmt=',', cmap='Blues', ax=axes[1], cbar_kws={'label':'N usuarios'})
axes[1].set_title('Nivel 1 × Nivel 2 (counts)')
axes[1].set_xlabel('Nivel 2 cluster')
axes[1].set_ylabel('Nivel 1 cluster')
plt.tight_layout()
plt.savefig(INFORMES / 'nivel1_x_nivel2_heatmap.png', dpi=120, bbox_inches='tight')
plt.close()
print('heatmap guardado')


Assignments: (114412, 4)
Crosstab (%):
n2_cluster       -99   -1    0    1    2    3     4 
nivel1_cluster                                      
0                0.0  57.7  0.0  0.0  0.0  0.0  42.3
1               52.2  17.2  0.2  0.1  0.1  0.3  30.0
2               82.8  16.0  0.0  0.4  0.3  0.5   0.2
3               87.4   9.6  0.1  0.3  0.3  0.2   1.9
4               97.8   1.2  0.2  0.1  0.1  0.2   0.4
5               87.7  10.3  0.1  0.4  0.3  0.3   0.9


heatmap guardado


In [2]:
# archetype_naming.md — propuesta automática + plantilla para refinar
md = ['# Propuesta de nombres para arquetipos', '',
      f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
      '',
      'Esta es una propuesta INICIAL basada en features distintivas. Refinar manualmente.',
      '', '## Nivel 1 (identidad)', '']

# Cargar estadísticas Nivel 1
n1_stats = pd.read_csv(INFORMES / 'nivel1' / 'archetype_statistics.csv')
n1_dist = assignments['nivel1_cluster'].value_counts().sort_index()
n1_t2_cov = assignments.groupby('nivel1_cluster')['has_tier2'].mean() * 100

# Hipótesis basadas en cobertura Tier 2 y tamaño
hypotheses_n1 = {}
for k in sorted(n1_dist.index):
    n = n1_dist[k]
    pct = n / len(assignments) * 100
    cov_t2 = n1_t2_cov.get(k, 0)
    # Top 5 distintivas positivas
    top_over = n1_stats[(n1_stats['cluster']==k) & (n1_stats['distinctiveness']>0)].head(5)['feature'].tolist()
    # Heurística de nombre
    if cov_t2 >= 95:
        hint = '"Hyper-activo / Hardcore competitivo"'
    elif cov_t2 >= 40:
        hint = '"Activo moderado"'
    elif cov_t2 <= 5 and pct > 30:
        hint = '"Casual dormido (mainstream)"'
    elif cov_t2 <= 20:
        hint = '"Veterano especializado / dormido"'
    else:
        hint = '"Perfil mixto"'
    hypotheses_n1[k] = (hint, top_over)
    md += [f'### Arquetipo {k} — Propuesta: {hint}', '',
           f'- **N**: {n:,} ({pct:.2f}%)',
           f'- **Cobertura Tier 2**: {cov_t2:.1f}%',
           f'- **Features sobre-representadas (top 5)**: {", ".join(f"`{x}`" for x in top_over)}',
           '', '_Refinar narrativa con `archetype_profiles.md` y `distinctive_features.md`._',
           '', '---', '']

md += ['', '## Nivel 2 (sub-arquetipos de actividad)', '']
n2_stats = pd.read_csv(INFORMES / 'nivel2' / 'subarchetype_statistics.csv')
n2_dist = assignments[assignments['nivel2_cluster'].notna()].copy()
n2_dist['n2'] = n2_dist['nivel2_cluster'].fillna(-1).astype(int)
n2_counts = n2_dist['n2'].value_counts().sort_index()

for k in sorted(n2_counts.index):
    n = n2_counts[k]
    label = 'OUTLIERS — "Patrón irregular"' if k==-1 else f'Sub-arquetipo {k}'
    top_over = n2_stats[(n2_stats['cluster']==k) & (n2_stats['distinctiveness']>0)].head(5)['feature'].tolist()
    md += [f'### {label}', '',
           f'- **N**: {n:,} ({n/len(n2_dist)*100:.2f}% de los activos)',
           f'- **Features sobre-representadas**: {", ".join(f"`{x}`" for x in top_over)}',
           '', '_(rellenar manualmente)_', '', '---', '']

(INFORMES / 'archetype_naming.md').write_text('\n'.join(md))
print('archetype_naming.md guardado')


archetype_naming.md guardado


In [3]:
# combined_summary.md
md = ['# Caracterización combinada de arquetipos — síntesis', '',
      f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
      f'**Total sample**: {len(assignments):,}',
      '',
      '## Síntesis Nivel 1 (identitario)', '',
      '| Arquetipo | N | % | Cobertura Tier 2 |',
      '|---:|---:|---:|---:|']
for k in sorted(n1_dist.index):
    md.append(f"| {k} | {n1_dist[k]:,} | {n1_dist[k]/len(assignments)*100:.2f}% | {n1_t2_cov[k]:.1f}% |")

md += ['', '## Síntesis Nivel 2 (actividad)', '',
       '| Sub-arquetipo | N | % de Tier 2 | % del total 114k |',
       '|---:|---:|---:|---:|']
for k in sorted(n2_counts.index):
    label = 'OUTLIERS' if k==-1 else str(k)
    md.append(f"| {label} | {n2_counts[k]:,} | {n2_counts[k]/len(n2_dist)*100:.2f}% | {n2_counts[k]/len(assignments)*100:.2f}% |")

md += ['', '## Matriz cruzada N1 × N2 (% por fila)', '',
       'Cada fila suma 100% — distribución del arquetipo identitario entre sub-arquetipos de actividad.',
       '', 'Ver `nivel1_x_nivel2_heatmap.png` y `nivel1_x_nivel2_crosstab_pct.csv`.',
       '', '## Implicaciones para diseño de contramedidas (Fase 7)', '',
       '- **Arquetipo 0 (100% activo)**: contramedidas predictivas basadas en sub-arquetipo de actividad',
       '- **Arquetipo 4 (mainstream casual, 59%)**: contramedidas genéricas de retención (notif daily, eventos abiertos)',
       '- **Arquetipos 1-3, 5 (especializados)**: refinar según features distintivas en `distinctive_features.md`']

(INFORMES / 'combined_summary.md').write_text('\n'.join(md))
print('combined_summary.md guardado')


combined_summary.md guardado
